# Principal Component Analysis: From First Principles

Principal Component Analysis (PCA) is a linear dimensionality-reduction method. It creates new numeric features, called principal components, by projecting the data onto directions that capture as much variance as possible. Keeping only the first few components gives a lower-dimensional representation that preserves as much of the original variance as those directions allow.

PCA is unsupervised: it uses the feature matrix, not a target label. It does not select a subset of original features; each component is usually a weighted combination of many original features.

## Why PCA Is Useful

Datasets can have many measured features, and some may carry overlapping information. For example, several measurements of size may move together. Representing all of them can make visualization difficult, increase storage and computation, or make a downstream model harder to fit. PCA can summarize correlated variation using fewer coordinates.

A two-dimensional cloud of points can often be described by a long direction and a short direction. If most variation lies along the long direction, projecting onto it preserves much of the spread with one number per row instead of two. In a dataset with many features, PCA generalizes this idea to find an ordered set of perpendicular directions.

PCA maximizes variance, not task relevance. A direction with high variance may reflect noise, a measurement scale, or an effect unrelated to a later prediction target. Dimensionality reduction should therefore be evaluated for the intended use, not assumed to improve every model.

## Principal Components and the Optimization Goal

A direction is a unit vector $\mathbf{w}$. Projecting a centered observation $\mathbf{x}$ onto that direction gives the scalar score $z=\mathbf{x}^T\mathbf{w}$. PCA chooses the first direction to maximize the variance of these scores:

$$
\mathbf{w}_1 = \underset{\|\mathbf{w}\|_2=1}{\operatorname{argmax}}\;\operatorname{Var}(X\mathbf{w})
$$

The unit-length constraint is essential: without it, scaling $\mathbf{w}$ would make the projected variance grow without bound. The solution is an eigenvector of the data's covariance matrix corresponding to its largest eigenvalue.

The second principal component maximizes projected variance subject to being perpendicular to the first. Later components are chosen the same way, each orthogonal to the preceding components. The eigenvalue associated with a component is the variance captured along that direction.

## Variance and Geometry

Variance measures how spread out values are around their mean. In PCA, the variance of the projected scores is used as a measure of how much the data varies along a candidate direction. PCA orders directions from greatest captured variance to least.

The first component captures the most variance possible along any single unit direction. The second captures the most remaining variance under the constraint that it is perpendicular to the first. This orthogonality makes the component scores uncorrelated on the centered training data, though uncorrelated does not mean statistically independent in general.

High variance is a useful compression criterion when important structure is expressed as broad linear variation. It is not a universal definition of information: a low-variance direction can still matter for a particular scientific or prediction task.

## Centering, Scaling, and the PCA Transformation

Suppose $X$ has $n$ rows and $d$ numeric features. PCA first centers each feature by subtracting its training-set mean. Write the centered matrix as $X_c$. Centering places the origin at the data mean, which is necessary for PCA to describe variation around that mean.

Whether to scale after centering depends on the measurement units and the goal. If features use very different units or ranges, standardizing each feature to unit variance prevents large-scale features from dominating. If the units are comparable and their relative variances are meaningful, scaling may discard useful information. Centering and scaling are separate decisions.

The sample covariance matrix of centered data is:

$$
\Sigma = \frac{1}{n-1}X_c^T X_c
$$

An eigendecomposition of $\Sigma$ gives orthonormal directions in the columns of $W$. Sort them by decreasing eigenvalue, then retain the first $k$ directions. The transformed scores are:

$$
Z = X_c W_k
$$

Here $W_k$ has shape $d\times k$ and $Z$ has shape $n\times k$. Each row of $Z$ contains one observation's coordinates in the new component system. In practice, PCA is often computed using Singular Value Decomposition (SVD) of $X_c$ rather than explicitly forming and decomposing the covariance matrix. SVD is numerically stable and also yields the component directions.

The sign of an eigenvector is arbitrary: $\mathbf{w}$ and $-\mathbf{w}$ describe the same axis, with scores whose signs are reversed.

## Worked Example: A Two-Feature Dataset

Consider four already-centered observations: $( -2,-1)$, $(-1,-2)$, $(1,2)$, and $(2,1)$. Their sample covariance matrix is:

$$
\Sigma = \frac{1}{3}\begin{bmatrix}10 & 8 \\ 8 & 10\end{bmatrix} = \begin{bmatrix}10/3 & 8/3 \\ 8/3 & 10/3\end{bmatrix}
$$

The eigenvectors point along $(1,1)$ and $(1,-1)$. The corresponding eigenvalues are $6$ and $2/3$, so the first component points along $\mathbf{w}_1=(1,1)/\sqrt{2}$. The total variance is the sum of the eigenvalues, $6+2/3=20/3$. Therefore, the first component's explained-variance ratio is:

$$
\frac{6}{20/3}=0.9=90\%
$$

For observation $(2,1)$, the first-component score is $(2,1)\cdot(1,1)/\sqrt{2}=3/\sqrt{2}$. Projecting back onto that one direction gives $(3/2,3/2)$. The squared reconstruction error is $(2-3/2)^2+(1-3/2)^2=1/2$. Keeping this component compresses two coordinates to one while retaining 90% of the dataset's variance in this example.

## Explained Variance and Choosing the Number of Components

Each component's eigenvalue is the variance of the centered data along that component. If the eigenvalues are $\lambda_1\geq\lambda_2\geq\cdots\geq\lambda_d$, the explained-variance ratio for component $j$ is:

$$
r_j = \frac{\lambda_j}{\sum_{\ell=1}^{d}\lambda_\ell}
$$

The cumulative ratio for the first $k$ components is $\sum_{j=1}^{k}r_j$. It reports the fraction of total training-set variance represented by those components. A scree plot shows the individual eigenvalues or ratios; an elbow can suggest where later components contribute less, but the elbow may not be clear.

Choose $k$ by balancing compression against information loss and downstream performance. A target such as 90% or 95% cumulative variance is a heuristic, not a universal rule. In supervised modeling, choose preprocessing and $k$ using training data and validation, never by fitting PCA to the full dataset before the train/test split. Otherwise, information from the test set leaks into the learned directions.

## When to Use PCA and Its Limitations

PCA is useful when a compact linear representation helps with visualization, compression, noise reduction, or reducing redundant numeric inputs. It can also make some downstream algorithms faster. When projecting to two or three components for visualization, remember that separation in the projection may not reflect all structure in the original feature space.

PCA has important trade-offs:

- Components are combinations of original features, so they can be harder to interpret than the original measurements.
- PCA is linear. It does not uncover curved or nonlinear structure; methods such as kernel PCA or nonlinear manifold methods address different goals and assumptions.
- High variance is not necessarily predictive or scientifically important, and low-variance directions may still matter.
- Scaling and outliers can strongly affect the directions. PCA is based on means and covariance, both of which are sensitive to extreme observations.
- Truncating components loses information. PCA does not guarantee better model accuracy.

For a fair workflow, split the data first, fit imputation, scaling, and PCA on the training set only, then use the fitted transformations on validation, test, or future data. Keep PCA inside a modeling pipeline when tuning component count. Use PCA when its compression or geometry suits the task; keep the original features when direct feature-level interpretation is essential.